# Figure 3: K562 held-out benchmark and ablation

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Data preparation, training, and loading provenance

Panels a-g use saved per-condition and aggregate metrics from the
three-seed K562 main experiment and matched baselines. Panel h uses the
three-seed full-prior/order ablations. The registered training,
generation, and metric entry points are audited below; this notebook
only reads frozen CSV results.

In [ ]:
panel_map = REGISTRY.loc[REGISTRY["figure"].eq("Fig3")].copy()
required = ["source_data", "training_code", "evaluation_code", "plot_code", "canonical_panel"]
display(panel_map[["panel", "panel_type", "experiment_id", "claim_or_role", "status"]])

def archived_paths_exist(value, base):
    if value == "NA":
        return True
    return all((base / item).exists() for item in str(value).split(";"))

for column in ["source_data", "plot_code", "canonical_panel"]:
    missing = [
        value for value in panel_map[column]
        if not archived_paths_exist(value, ARCHIVE)
    ]
    assert not missing, f"Missing {column}: {missing}"
print("Panel-level figure inputs and plotting assets are present.")

In [ ]:
source = SOURCE_DATA / "Fig3"
per_condition = pd.read_csv(source / "k562_internal_heldout_per_condition_oldstyle.csv")
aggregate = pd.read_csv(source / "k562_internal_heldout_comparison_metrics.csv")
ablations = pd.read_csv(source / "k562_current_ablation_seed_metrics.csv")
print("Per-condition rows:", len(per_condition))
display(aggregate)
display(ablations.groupby("variant").size().rename("rows"))

In [ ]:
subprocess.run(
    [sys.executable, str(ARCHIVE / "scripts" / "Fig3" / "make_figure3_180mm_publication.py")],
    check=True,
)
subprocess.run(
    [sys.executable, str(ARCHIVE / "scripts" / "Fig3" / "make_k562_ablation_bar_strip_20260716.py")],
    check=True,
)

In [ ]:
outputs = [
    ("a", "Per-condition perturbation direction", REPRO / "Fig3" / "figure3a_delta_pcc.svg"),
    ("b", "Per-condition response-gene recovery", REPRO / "Fig3" / "figure3b_topk_de.svg"),
    ("c", "Per-condition expression-range coverage", REPRO / "Fig3" / "figure3c_pra_all.svg"),
    ("d", "Per-condition gene correlation structure", REPRO / "Fig3" / "figure3d_csa_pearson.svg"),
    ("e", "GGE Wasserstein distance", REPRO / "Fig3" / "figure3e_gge_wasserstein.svg"),
    ("f", "GGE MMD", REPRO / "Fig3" / "figure3f_gge_mmd.svg"),
    ("g", "GGE energy distance", REPRO / "Fig3" / "figure3g_gge_energy.svg"),
    ("h", "K562 ablation analysis", REPRO / "Fig3" / "figure4h_k562_ablation_bar_strip_20260716.svg"),
]
assert all(path.exists() for _, _, path in outputs)
for panel, title, path in outputs:
    display(Markdown(f"### Fig. 3{panel}: {title}"))
    display(SVG(filename=str(path)))